<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/pytorch_inf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("gpt2")

model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float16
)

model = model.to(device)

model.eval()

inputs = tokenizer(
    "Explain the transformer in simple terms",
    return_tensors="pt"
)

inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.inference_mode():

  outputs = model(**inputs)

  logits = outputs.logits

  probs = torch.softmax(logits[:,-1,:], dim=-1)

  next_token = torch.argmax(probs, dim=-1)

  print("Next token ID:", next_token.item())


  generated_ids = model.generate(
      **inputs,
      max_new_tokens=50
  )

  response = tokenizer.decode(
      generated_ids[0],
      skip_special_tokens=True
  )

  print(response)

  torch.cuda.empty_cache()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Next token ID: 13
Explain the transformer in simple terms.

The transformer is a simple, non-trivial, non-trivial, non-trivial, non-trivial, non-trivial, non-trivial, non-trivial


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype
).to(device)

model.eval()


if device == "cuda":
    model = torch.compile(model, mode="reduce-overhead")

prompts = [
    "Explain attention in transformers simply.",
    "What is LLM inference?",
    "Why do we use GPUs for deep learning?"
]

inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

responses = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

for i, response in enumerate(responses, 1):
    print(f"\nResponse {i}:\n{response}")

if device == "cuda":
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



Response 1:
Explain attention in transformers simply..

If you are using an object that is a string, use the transformers.transformers() method.

In a list, use the transformers.transformers() method.

In a list of numbers, use the transformers.transformers() method.



Response 2:
What is LLM inference? a "statistical inference"? A statistical inference is when you find something that is hard to understand.

Why is it important to know when to use a statistical inference?

We know that many statistical inference techniques require that we use the same parameters for the data. In other words, we

Response 3:
Why do we use GPUs for deep learning?

As a general rule, GPUs are not good for deep learning because they require very few resources. This is because GPU-based applications are expensive to implement and require very few compute units to implement. However, GPUs can also be used for other kinds of deep learning applications as well.




# distributed training

from torch.nn.parallel import DistributedDataParallel as DDP

model = DDP(model, device_ids=[local_rank])